In [ ]:
import os
import time
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from xgboost import XGBClassifier

# ===== Config =====
SEED = 42
rng = np.random.default_rng(SEED)
N_SPLITS = 5
N_MODELS = 100
MAX_HOURS = 2.5
USE_EARLY_STOPPING = True
EARLY_STOPPING_ROUNDS = 100
WEIGHT_SEARCH_MAX_TRIALS = 400

# ===== Runtime Clock =====
overall_start = time.perf_counter()

def elapsed_seconds():
    return time.perf_counter() - overall_start

print(f'Stacker search budget configured: {MAX_HOURS:.2f}h')

# ===== Paths =====
local_train = '/workspaces/KagglePredictingStellarClass/data/train.csv'
local_test = '/workspaces/KagglePredictingStellarClass/data/test.csv'
local_sample = '/workspaces/KagglePredictingStellarClass/data/sample_submission.csv'
local_fine = '/workspaces/KagglePredictingStellarClass/xgb_fine_random_search_results.csv'

kaggle_train = '/kaggle/input/competitions/playground-series-s6e6/train.csv'
kaggle_test = '/kaggle/input/competitions/playground-series-s6e6/test.csv'
kaggle_sample = '/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv'
kaggle_fine_candidates = [
    '/kaggle/input/notebooks/diracturing/finesearch/fine_search_outputs/xgb_fine_random_search_results.csv',
    '/kaggle/working/fine_search_outputs/xgb_fine_random_search_results.csv',
    '/kaggle/working/xgb_fine_random_search_results.csv'
]

train_path = local_train if os.path.exists(local_train) else kaggle_train
test_path = local_test if os.path.exists(local_test) else kaggle_test
sample_path = local_sample if os.path.exists(local_sample) else kaggle_sample
if os.path.exists(local_fine):
    fine_path = local_fine
else:
    fine_path = None
    for p in kaggle_fine_candidates:
        if os.path.exists(p):
            fine_path = p
            break

if fine_path is None:
    raise FileNotFoundError('Could not find xgb_fine_random_search_results.csv in local or Kaggle paths.')

for p in [train_path, test_path, sample_path, fine_path]:
    if not os.path.exists(p):
        raise FileNotFoundError(f'Missing required file: {p}')

print('train_path:', train_path)
print('test_path:', test_path)
print('sample_path:', sample_path)
print('fine_path:', fine_path)

out_dir = '/kaggle/working/final_models_outputs' if os.path.exists('/kaggle/working') else '/workspaces/KagglePredictingStellarClass/final_models_outputs'
os.makedirs(out_dir, exist_ok=True)
selected_models_path = os.path.join(out_dir, '01_selected_base_models.csv')
progress_path = os.path.join(out_dir, '02_base_progress.csv')
strategy_scores_path = os.path.join(out_dir, '03_strategy_scores.csv')
weights_path = os.path.join(out_dir, '04_weighted_blend_best.json')
stacker_trials_path = os.path.join(out_dir, '05_stacker_trials.csv')
winner_path = os.path.join(out_dir, '06_winner.json')
submission_path = os.path.join(out_dir, '07_submission.csv')

# ===== Load Data =====
df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)
df_sample = pd.read_csv(sample_path)
df_fine = pd.read_csv(fine_path)

required_fine_cols = [
    'mean_balanced_accuracy', 'n_estimators', 'learning_rate', 'max_depth',
    'min_child_weight', 'subsample', 'colsample_bytree', 'reg_alpha', 'reg_lambda', 'gamma'
]
missing_fine = [c for c in required_fine_cols if c not in df_fine.columns]
if missing_fine:
    raise ValueError(f'Fine search CSV missing columns: {missing_fine}')

required_train_cols = ['id', 'alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type', 'galaxy_population', 'class']
required_test_cols = ['id', 'alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type', 'galaxy_population']
required_sample_cols = ['id', 'class']
for col in required_train_cols:
    if col not in df_train.columns:
        raise ValueError(f'train.csv missing column: {col}')
for col in required_test_cols:
    if col not in df_test.columns:
        raise ValueError(f'test.csv missing column: {col}')
for col in required_sample_cols:
    if col not in df_sample.columns:
        raise ValueError(f'sample_submission.csv missing column: {col}')

# ===== Preprocessing (same feature engineering style) =====
def feature_engineering(df):
    out = df.copy()
    if 'id' in out.columns:
        out = out.drop(columns=['id'])
    out['u_g'] = out['u'] - out['g']
    out['g_r'] = out['g'] - out['r']
    out['r_i'] = out['r'] - out['i']
    out['i_z'] = out['i'] - out['z']
    out = out.drop(columns=['u', 'g', 'r', 'i', 'z'])
    return out

train_eng = feature_engineering(df_train)
test_eng = feature_engineering(df_test)

cat_cols = ['spectral_type', 'galaxy_population']
cat_maps = {}
for col in cat_cols:
    vals = train_eng[col].astype(str).fillna('NA')
    uniques = pd.Index(sorted(vals.unique()))
    mapping = {v: i for i, v in enumerate(uniques)}
    cat_maps[col] = mapping
    train_eng[col] = vals.map(mapping).astype(np.int32)
    test_vals = test_eng[col].astype(str).fillna('NA')
    test_eng[col] = test_vals.map(mapping).fillna(-1).astype(np.int32)

y_encoder = LabelEncoder()
y = y_encoder.fit_transform(train_eng['class'].astype(str))
train_eng = train_eng.drop(columns=['class'])

X = train_eng
X_test = test_eng
num_class = len(np.unique(y))

print('X shape:', X.shape)
print('X_test shape:', X_test.shape)
print('num_class:', num_class)

# ===== Select top-N fine models =====
param_cols = ['n_estimators', 'learning_rate', 'max_depth', 'min_child_weight', 'subsample', 'colsample_bytree', 'reg_alpha', 'reg_lambda', 'gamma']
top_df = df_fine.sort_values('mean_balanced_accuracy', ascending=False).head(N_MODELS).copy().reset_index(drop=True)
top_df.to_csv(selected_models_path, index=False)
print(f'Selected {len(top_df)} base models (requested N_MODELS={N_MODELS})')

# ===== OOF + test predictions for base models =====
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
n_train = X.shape[0]
n_test = X_test.shape[0]

base_oof_list = []
base_test_list = []
base_meta_rows = []

common = dict(
    objective='multi:softprob',
    num_class=num_class,
    eval_metric='mlogloss',
    tree_method='hist',
    random_state=SEED
)
if USE_EARLY_STOPPING:
    common['early_stopping_rounds'] = EARLY_STOPPING_ROUNDS

print('Starting base-model OOF generation...')

for model_idx in range(len(top_df)):
    row = top_df.iloc[model_idx]
    params = {
        'n_estimators': int(row['n_estimators']),
        'learning_rate': float(row['learning_rate']),
        'max_depth': int(row['max_depth']),
        'min_child_weight': float(row['min_child_weight']),
        'subsample': float(row['subsample']),
        'colsample_bytree': float(row['colsample_bytree']),
        'reg_alpha': float(row['reg_alpha']),
        'reg_lambda': float(row['reg_lambda']),
        'gamma': float(row['gamma']),
        'n_jobs': -1
    }

    oof_probs = np.zeros((n_train, num_class), dtype=np.float32)
    test_fold_probs = np.zeros((N_SPLITS, n_test, num_class), dtype=np.float32)
    fold_scores = []
    fold_fit_times = []

    model_start = time.perf_counter()

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_tr = X.iloc[tr_idx]
        y_tr = y[tr_idx]
        X_va = X.iloc[va_idx]
        y_va = y[va_idx]

        fit_start = time.perf_counter()
        try:
            model = XGBClassifier(device='cuda', **params, **common)
            if USE_EARLY_STOPPING:
                model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
            else:
                model.fit(X_tr, y_tr, verbose=False)
        except Exception:
            model = XGBClassifier(device='cpu', **params, **common)
            if USE_EARLY_STOPPING:
                model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
            else:
                model.fit(X_tr, y_tr, verbose=False)
        fold_fit_times.append(time.perf_counter() - fit_start)

        va_prob = model.predict_proba(X_va).astype(np.float32)
        oof_probs[va_idx] = va_prob
        va_pred = np.argmax(va_prob, axis=1)
        fold_scores.append(float(balanced_accuracy_score(y_va, va_pred)))

        test_prob = model.predict_proba(X_test).astype(np.float32)
        test_fold_probs[fold] = test_prob

    model_time = time.perf_counter() - model_start
    mean_fold_score = float(np.mean(fold_scores))

    base_oof_list.append(oof_probs)
    base_test_list.append(np.mean(test_fold_probs, axis=0).astype(np.float32))

    meta_row = {
        'base_model_rank': model_idx + 1,
        'mean_balanced_accuracy_oof': mean_fold_score,
        'std_balanced_accuracy_oof': float(np.std(fold_scores)),
        'model_time_sec': float(model_time),
        'mean_fold_fit_sec': float(np.mean(fold_fit_times)),
        **params
    }
    base_meta_rows.append(meta_row)

    # Save progress at end of base model loop
    pd.DataFrame(base_meta_rows).to_csv(progress_path, index=False)

    print(
        f'Base model {model_idx + 1}/{len(top_df)} | '
        f'OOF bal_acc={mean_fold_score:.6f} | '
        f'model_time={model_time:.1f}s | '
        f'elapsed={elapsed_seconds()/3600:.2f}h'
    )

if len(base_oof_list) == 0:
    raise RuntimeError('No complete base models were trained. Check data paths and model parameters.')

pd.DataFrame(base_meta_rows).to_csv(progress_path, index=False)
n_base = len(base_oof_list)
print(f'Completed base models: {n_base}')

# Convert to tensors for fast blending
oof_cube = np.stack(base_oof_list, axis=0).astype(np.float32)   # (m, n_train, num_class)
test_cube = np.stack(base_test_list, axis=0).astype(np.float32) # (m, n_test, num_class)

# ===== Strategy 1: Plain Average =====
plain_oof = oof_cube.mean(axis=0)
plain_oof = plain_oof / np.clip(plain_oof.sum(axis=1, keepdims=True), 1e-9, None)
plain_pred = np.argmax(plain_oof, axis=1)
plain_score = float(balanced_accuracy_score(y, plain_pred))

plain_test = test_cube.mean(axis=0)
plain_test = plain_test / np.clip(plain_test.sum(axis=1, keepdims=True), 1e-9, None)

print(f'Plain average OOF balanced_accuracy: {plain_score:.6f}')

# ===== Strategy 2: Weighted Blend (Dirichlet random search) =====
weighted_best_score = -1.0
weighted_best_weights = np.ones(n_base, dtype=np.float64) / n_base
weighted_trials = 0

while weighted_trials < WEIGHT_SEARCH_MAX_TRIALS:
    if weighted_trials == 0:
        w = np.ones(n_base, dtype=np.float64) / n_base
    else:
        w = rng.dirichlet(np.ones(n_base, dtype=np.float64))

    blend = np.tensordot(w, oof_cube, axes=(0, 0))
    blend = blend / np.clip(blend.sum(axis=1, keepdims=True), 1e-9, None)
    pred = np.argmax(blend, axis=1)
    score = float(balanced_accuracy_score(y, pred))

    if score > weighted_best_score:
        weighted_best_score = score
        weighted_best_weights = w.copy()

    weighted_trials += 1

weighted_oof = np.tensordot(weighted_best_weights, oof_cube, axes=(0, 0))
weighted_oof = weighted_oof / np.clip(weighted_oof.sum(axis=1, keepdims=True), 1e-9, None)
weighted_test = np.tensordot(weighted_best_weights, test_cube, axes=(0, 0))
weighted_test = weighted_test / np.clip(weighted_test.sum(axis=1, keepdims=True), 1e-9, None)

print(f'Weighted blend OOF balanced_accuracy: {weighted_best_score:.6f} (trials={weighted_trials})')

with open(weights_path, 'w', encoding='utf-8') as f:
    json.dump(
        {
            'weighted_best_score': weighted_best_score,
            'weighted_trials': weighted_trials,
            'weights': weighted_best_weights.tolist()
        },
        f,
        indent=2
    )

# ===== Strategy 3: XGB Stacker (search budget only) =====
# Select top N models for all ensemble strategies
n_ensemble = min(N_MODELS, n_base)
base_scores_df = pd.DataFrame(base_meta_rows).sort_values('mean_balanced_accuracy_oof', ascending=False).reset_index(drop=True)
ensemble_indices = base_scores_df.head(n_ensemble)['base_model_rank'].to_numpy() - 1

stack_oof = oof_cube[ensemble_indices].transpose(1, 0, 2).reshape(n_train, n_ensemble * num_class).astype(np.float32)
stack_test = test_cube[ensemble_indices].transpose(1, 0, 2).reshape(n_test, n_ensemble * num_class).astype(np.float32)

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
tr_meta_idx, va_meta_idx = next(splitter.split(stack_oof, y))
X_meta_tr, y_meta_tr = stack_oof[tr_meta_idx], y[tr_meta_idx]
X_meta_va, y_meta_va = stack_oof[va_meta_idx], y[va_meta_idx]

stack_best_score = -1.0
stack_best_params = None
stack_best_model = None
stack_trials = []

def sample_stacker_params(rng_local):
    return {
        'n_estimators': int(rng_local.integers(300, 2200)),
        'learning_rate': float(np.exp(rng_local.uniform(np.log(0.01), np.log(0.20)))),
        'max_depth': int(rng_local.integers(3, 9)),
        'min_child_weight': float(np.exp(rng_local.uniform(np.log(0.5), np.log(10.0)))),
        'subsample': float(rng_local.uniform(0.6, 1.0)),
        'colsample_bytree': float(rng_local.uniform(0.6, 1.0)),
        'reg_alpha': float(np.exp(rng_local.uniform(np.log(1e-6), np.log(3.0)))),
        'reg_lambda': float(np.exp(rng_local.uniform(np.log(1e-6), np.log(5.0)))),
        'gamma': float(rng_local.uniform(0.0, 2.0)),
        'n_jobs': -1
    }

stacker_budget_seconds = MAX_HOURS * 3600.0
stacker_search_start = time.perf_counter()
stacker_deadline = stacker_search_start + stacker_budget_seconds

def stacker_remaining_seconds():
    return max(0.0, stacker_deadline - time.perf_counter())

def within_stacker_budget():
    return stacker_remaining_seconds() > 0.0

stack_trial_idx = 0
while stack_trial_idx == 0 or within_stacker_budget():
    params = sample_stacker_params(rng)
    trial_start = time.perf_counter()

    stack_common = dict(
        objective='multi:softprob',
        num_class=num_class,
        eval_metric='mlogloss',
        tree_method='hist',
        random_state=SEED
    )
    if USE_EARLY_STOPPING:
        stack_common['early_stopping_rounds'] = EARLY_STOPPING_ROUNDS

    try:
        stack_model = XGBClassifier(device='cuda', **params, **stack_common)
        if USE_EARLY_STOPPING:
            stack_model.fit(X_meta_tr, y_meta_tr, eval_set=[(X_meta_va, y_meta_va)], verbose=False)
        else:
            stack_model.fit(X_meta_tr, y_meta_tr, verbose=False)
    except Exception:
        stack_model = XGBClassifier(device='cpu', **params, **stack_common)
        if USE_EARLY_STOPPING:
            stack_model.fit(X_meta_tr, y_meta_tr, eval_set=[(X_meta_va, y_meta_va)], verbose=False)
        else:
            stack_model.fit(X_meta_tr, y_meta_tr, verbose=False)

    va_prob = stack_model.predict_proba(X_meta_va)
    va_pred = np.argmax(va_prob, axis=1)
    score = float(balanced_accuracy_score(y_meta_va, va_pred))

    row = {
        'trial': stack_trial_idx + 1,
        'score': score,
        'trial_time_sec': float(time.perf_counter() - trial_start),
        **params
    }
    stack_trials.append(row)

    if score > stack_best_score:
        stack_best_score = score
        stack_best_params = params
        stack_best_model = stack_model

    stack_trial_idx += 1

    if stack_trial_idx % 10 == 0 or stack_trial_idx == 1:
        print(
            f'Stacker trial {stack_trial_idx} | best_score={stack_best_score:.6f} | '
            f'elapsed={elapsed_seconds()/3600:.2f}h | stacker_remaining={stacker_remaining_seconds()/3600:.2f}h'
        )
        pd.DataFrame(stack_trials).to_csv(stacker_trials_path, index=False)

if stack_trials:
    pd.DataFrame(stack_trials).to_csv(stacker_trials_path, index=False)

# Build full OOF/test predictions for best stacker if available
stack_full_oof = None
stack_full_test = None
if stack_best_model is not None:
    # Refit best stacker on full OOF feature matrix for final prediction
    stack_common_final = dict(
        objective='multi:softprob',
        num_class=num_class,
        eval_metric='mlogloss',
        tree_method='hist',
        random_state=SEED
    )
    # Refit on full OOF data without early stopping (use all training data for final model)
    try:
        stack_final = XGBClassifier(device='cuda', **stack_best_params, **stack_common_final)
        stack_final.fit(stack_oof, y, verbose=False)
    except Exception:
        stack_final = XGBClassifier(device='cpu', **stack_best_params, **stack_common_final)
        stack_final.fit(stack_oof, y, verbose=False)
    stack_full_oof = stack_final.predict_proba(stack_oof)
    stack_full_oof = stack_full_oof / np.clip(stack_full_oof.sum(axis=1, keepdims=True), 1e-9, None)
    stack_full_pred = np.argmax(stack_full_oof, axis=1)
    stack_oof_score = float(balanced_accuracy_score(y, stack_full_pred))

    stack_full_test = stack_final.predict_proba(stack_test)
    stack_full_test = stack_full_test / np.clip(stack_full_test.sum(axis=1, keepdims=True), 1e-9, None)
else:
    stack_oof_score = -1.0

print(f'Stacker best validation score: {stack_best_score:.6f} | full OOF score: {stack_oof_score:.6f} | trials={len(stack_trials)}')

# ===== Compare strategies =====
strategy_rows = [
    {'strategy': 'plain_average', 'oof_balanced_accuracy': plain_score, 'details': f'base_models={n_base}'},
    {'strategy': 'weighted_blend', 'oof_balanced_accuracy': weighted_best_score, 'details': f'base_models={n_base}, trials={weighted_trials}'},
    {'strategy': 'xgb_stacker', 'oof_balanced_accuracy': stack_oof_score, 'details': f'stack_models={n_ensemble}, trials={len(stack_trials)}'}
]
strategy_df = pd.DataFrame(strategy_rows).sort_values('oof_balanced_accuracy', ascending=False).reset_index(drop=True)
strategy_df.to_csv(strategy_scores_path, index=False)
display(strategy_df)

winner = strategy_df.iloc[0]['strategy']
winner_score = float(strategy_df.iloc[0]['oof_balanced_accuracy'])

if winner == 'plain_average':
    final_test_prob = plain_test
elif winner == 'weighted_blend':
    final_test_prob = weighted_test
else:
    if stack_full_test is None:
        # fallback safety
        winner = 'weighted_blend'
        winner_score = weighted_best_score
        final_test_prob = weighted_test
    else:
        final_test_prob = stack_full_test

with open(winner_path, 'w', encoding='utf-8') as f:
    json.dump(
        {
            'winner': winner,
            'winner_oof_balanced_accuracy': winner_score,
            'n_base_models_completed': n_base,
            'time_elapsed_hours': elapsed_seconds() / 3600.0
        },
        f,
        indent=2
    )

print(f'Winner: {winner} | OOF balanced_accuracy={winner_score:.6f}')

# ===== Submission =====
test_pred_idx = np.argmax(final_test_prob, axis=1)
test_pred_label = y_encoder.inverse_transform(test_pred_idx)

submission = df_sample.copy()
submission['id'] = df_test['id'].values
submission['class'] = test_pred_label
submission = submission[['id', 'class']]
submission.to_csv(submission_path, index=False)

print('Saved selected models:', selected_models_path)
print('Saved base progress:', progress_path)
print('Saved strategy scores:', strategy_scores_path)
print('Saved weighted blend details:', weights_path)
print('Saved stacker trials:', stacker_trials_path)
print('Saved winner details:', winner_path)
print('Saved submission:', submission_path)
print(f'Total elapsed: {elapsed_seconds()/60:.2f} minutes')
display(submission.head())